# FinSearch: Finance-Aware Chatbot
**Pipeline:** Hybrid Retrieval -> Groq LLM Re-ranking -> Answer Generation -> NLI Confidence Score

In [56]:
import sys, os
from pathlib import Path

_root = Path().resolve()
for _ in range(5):
    if (_root / 'config.py').exists():
        break
    _root = _root.parent

os.chdir(_root)
sys.path.insert(0, str(_root))

from config import FIQA_CORPUS, FIQA_QUERIES, FIQA_QRELS_TEST

DENSE_INDEX_DIR = os.path.join('dense_retrieval', 'Results', 'indexes')
FINRERANK_DIR   = os.path.join('finrerank', 'Results')

print('Repo root :', _root)
print('Index dir :', DENSE_INDEX_DIR)


Repo root : /Users/momo/Desktop/GEN AI/Finsearch Project/FinSearch_Intent_Aware_Financial_Document_Intelligence-
Index dir : dense_retrieval/Results/indexes


In [57]:
# Uncomment if not installed
# !pip install openai sentence-transformers faiss-cpu rank_bm25 pytrec_eval --user


In [58]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import pandas as pd
import numpy as np
import json, re, time, pickle
import pytrec_eval
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from tqdm import tqdm

print("Imports OK")


Imports OK


In [59]:
print('Loading corpus...')
corpus_df = pd.read_csv(FIQA_CORPUS, usecols=['_id', 'text'])
corpus_df = corpus_df.dropna(subset=['text']).reset_index(drop=True)
corpus_df['_id'] = corpus_df['_id'].astype(str)
corpus_id_to_text = dict(zip(corpus_df['_id'], corpus_df['text']))
print('Corpus:', len(corpus_id_to_text), 'passages')

queries_df = pd.read_csv(FIQA_QUERIES, usecols=['_id', 'text'])
queries_df['_id'] = queries_df['_id'].astype(str)
query_id_to_text = dict(zip(queries_df['_id'], queries_df['text']))

qrels_df = pd.read_csv(FIQA_QRELS_TEST)
qrels_df.columns = [c.strip() for c in qrels_df.columns]
qrels_df = qrels_df.astype(str)
query_col  = [c for c in qrels_df.columns if 'query'  in c.lower()][0]
corpus_col = [c for c in qrels_df.columns if 'corpus' in c.lower()][0]
score_col  = [c for c in qrels_df.columns if 'score'  in c.lower()][0]
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row[query_col])
    did = str(row[corpus_col])
    if qid not in qrels_dict:
        qrels_dict[qid] = {}
    qrels_dict[qid][did] = int(float(row[score_col]))
test_qids = list(qrels_dict.keys())
test_queries_df = queries_df[queries_df['_id'].isin(test_qids)].reset_index(drop=True)
print('Test queries:', len(test_queries_df), '| Qrels:', len(qrels_dict))


Loading corpus...
Corpus: 57600 passages
Test queries: 648 | Qrels: 648


In [60]:
faiss_path = os.path.join(DENSE_INDEX_DIR, 'faiss_minilm.index')
ids_path   = os.path.join(DENSE_INDEX_DIR, 'corpus_ids.pkl')
index      = faiss.read_index(faiss_path)
with open(ids_path, 'rb') as f:
    corpus_ids = pickle.load(f)
print('FAISS index:', index.ntotal, 'vectors')

dense_model = SentenceTransformer('all-MiniLM-L6-v2')
print('Dense model loaded')

def tokenize(text):
    t = str(text).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return t.split()

print('Building BM25 (1-2 min)...')
tokenized_corpus = [tokenize(t) for t in tqdm(corpus_df['text'])]
bm25 = BM25Okapi(tokenized_corpus)
print('BM25 ready')


FAISS index: 57600 vectors


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6056.03it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dense model loaded
Building BM25 (1-2 min)...


100%|██████████| 57600/57600 [00:00<00:00, 73421.70it/s]


BM25 ready


In [61]:
ALPHA = 0.7
TOP_K = 20

print('Encoding queries...')
query_texts = test_queries_df['text'].tolist()
query_ids   = test_queries_df['_id'].tolist()
query_embs  = dense_model.encode(query_texts, normalize_embeddings=True,
                                  batch_size=64, show_progress_bar=True)

print('Running Hybrid retrieval (top-20)...')
hybrid_top20 = {}
for i, (qid, q_text) in enumerate(tqdm(zip(query_ids, query_texts), total=len(query_ids))):
    q_vec = query_embs[i].reshape(1, -1).astype('float32')
    D, I  = index.search(q_vec, TOP_K * 5)
    dense_scores = {corpus_ids[idx]: float(D[0][j]) for j, idx in enumerate(I[0]) if idx < len(corpus_ids)}
    toks         = tokenize(q_text)
    bm25_raw     = bm25.get_scores(toks)
    top_bm25_idx = np.argsort(bm25_raw)[::-1][:TOP_K * 5]
    bm25_scores  = {str(corpus_df.iloc[idx]['_id']): float(bm25_raw[idx]) for idx in top_bm25_idx}
    all_docs = set(dense_scores) | set(bm25_scores)
    d_vals   = np.array([dense_scores.get(d, 0.0) for d in all_docs])
    b_vals   = np.array([bm25_scores.get(d,  0.0) for d in all_docs])
    d_norm   = (d_vals - d_vals.min()) / (d_vals.max() - d_vals.min() + 1e-9)
    b_norm   = (b_vals - b_vals.min()) / (b_vals.max() - b_vals.min() + 1e-9)
    hybrid   = ALPHA * d_norm + (1 - ALPHA) * b_norm
    ranked   = sorted(zip(all_docs, hybrid), key=lambda x: x[1], reverse=True)[:TOP_K]
    hybrid_top20[qid] = {doc_id: float(score) for doc_id, score in ranked}
print('Hybrid done:', len(hybrid_top20), 'queries')


Encoding queries...


Batches: 100%|██████████| 11/11 [00:00<00:00, 14.26it/s]


Running Hybrid retrieval (top-20)...


100%|██████████| 648/648 [01:26<00:00,  7.53it/s]

Hybrid done: 648 queries


In [62]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads .env from repo root

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY not found in .env file")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "meta-llama/llama-3.3-70b-instruct"

# Smoke test
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Reply with the single word: ready"}],
    max_tokens=5,
)
print("OpenRouter response:", resp.choices[0].message.content.strip())
print("OpenRouter + Llama 3.3 70B ready")

OpenRouter response: ready
OpenRouter + Llama 3.3 70B ready


In [63]:
def groq_rerank(query, candidates, corpus_id_to_text, client, top_k=5):
    lines = []
    for i, d in enumerate(candidates, 1):
        text = corpus_id_to_text.get(d, "")[:400].replace("\n", " ")
        lines.append(str(i) + ". " + text)
    n = len(candidates)
    passages = "\n".join(lines)
    prompt = (
        "You are a financial information retrieval expert.\n\n"
        "Query: \"" + query + "\"\n\n"
        "Below are " + str(n) + " candidate passages. "
        "Return a JSON array of passage numbers (1-based) sorted MOST to LEAST relevant. "
        "Include ALL " + str(n) + " numbers.\n\n"
        "Passages:\n" + passages + "\n\n"
        "Respond ONLY with a JSON array, e.g.: [3,1,5,2,4]"
    )
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150, temperature=0,
        )
        raw = r.choices[0].message.content.strip()
        m = re.search(r"\\[[\\d,\\s]+\\]", raw)
        if m:
            ranking = json.loads(m.group())
            ranking = [x for x in ranking if 1 <= x <= n]
            seen = set(ranking)
            for i in range(1, n+1):
                if i not in seen:
                    ranking.append(i)
            return {candidates[idx-1]: float(n - pos + 1) for pos, idx in enumerate(ranking[:top_k], 1)}
    except Exception:
        pass
    return {d: float(n - i) for i, d in enumerate(candidates[:top_k])}

print("groq_rerank() defined")


groq_rerank() defined


In [64]:
def groq_answer(query, top_docs, corpus_id_to_text, client):
    parts = []
    for i, d in enumerate(top_docs, 1):
        parts.append("[Doc " + str(i) + "] " + corpus_id_to_text.get(d, "")[:500])
    context = "\n".join(parts)
    prompt = (
        "You are an expert financial advisor assistant.\n"
        "Answer the question using ONLY the provided documents.\n"
        "If the documents do not contain enough information, say so.\n\n"
        "Question: " + query + "\n\n"
        "Documents:\n" + context + "\n\n"
        "Provide a clear, accurate financial answer based strictly on the documents above."
    )
    try:
        r = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300, temperature=0.1,
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return "Error: " + str(e)

print("groq_answer() defined")


groq_answer() defined


In [65]:
print("Loading NLI confidence model...")
nli_model = CrossEncoder("cross-encoder/nli-deberta-v3-small",
                         default_activation_function=None)
print("NLI model loaded")

def compute_confidence(answer, top_docs, corpus_id_to_text, hybrid_scores, nli_model):
    # Score 1: Retrieval confidence
    scores = [hybrid_scores.get(d, 0.0) for d in top_docs]
    s_min, s_max = min(scores), max(scores)
    norm = [(s - s_min) / (s_max - s_min + 1e-9) for s in scores]
    retrieval_conf = float(np.mean(norm))

    # Score 2: Faithfulness via NLI entailment
    # cross-encoder/nli-deberta-v3-small label order:
    #   index 0 = contradiction, index 1 = neutral, index 2 = entailment  ← FIXED
    entailment_scores = []
    for doc_id in top_docs:
        doc_text = corpus_id_to_text.get(doc_id, "")[:500]
        if not doc_text.strip():
            continue
        logits = nli_model.predict([[doc_text, answer]])
        probs  = np.exp(logits) / np.exp(logits).sum(axis=1, keepdims=True)
        entailment_scores.append(float(probs[0][2]))  # index 2 = entailment (FIXED from 1)
    faithfulness_conf = float(np.mean(entailment_scores)) if entailment_scores else 0.0

    final_conf = round(0.4 * retrieval_conf + 0.6 * faithfulness_conf, 4)
    return {
        "retrieval_confidence":    round(retrieval_conf,    4),
        "faithfulness_confidence": round(faithfulness_conf, 4),
        "final_confidence":        final_conf
    }

print("compute_confidence() defined  [NLI entailment index FIXED: 1 -> 2]")

The CrossEncoder `default_activation_function` argument was renamed and is now deprecated, please use `activation_fn` instead.


Loading NLI confidence model...


Loading weights: 100%|██████████| 106/106 [00:00<00:00, 8719.28it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NLI model loaded
compute_confidence() defined  [NLI entailment index FIXED: 1 -> 2]


In [66]:
# Demo on 5 sample queries
SAMPLE_QIDS = test_queries_df["_id"].head(5).tolist()
RATE_DELAY  = 2.5
demo_results = []

for qid in SAMPLE_QIDS:
    query      = query_id_to_text.get(qid, "")
    candidates = list(hybrid_top20[qid].keys())

    # Step 1: Groq re-ranks 20 -> top-5
    reranked  = groq_rerank(query, candidates, corpus_id_to_text, client, top_k=5)
    top5_docs = list(reranked.keys())
    time.sleep(RATE_DELAY)

    # Step 2: Groq generates financial answer
    answer = groq_answer(query, top5_docs, corpus_id_to_text, client)
    time.sleep(RATE_DELAY)

    # Step 3: True confidence score via NLI
    conf  = compute_confidence(answer, top5_docs, corpus_id_to_text, hybrid_top20[qid], nli_model)
    fc    = conf["final_confidence"]
    label = "HIGH - Answer confidently" if fc >= 0.7 else ("MEDIUM - Answer with caveat" if fc >= 0.4 else "LOW - Cannot answer reliably")

    demo_results.append({"query_id": qid, "query": query, "answer": answer,
                         "top5_docs": top5_docs, **conf, "conf_label": label})

    print("=" * 70)
    print("Query           :", query)
    print("Answer          :", answer[:300])
    print("Retrieval Conf  :", conf["retrieval_confidence"])
    print("Faithfulness    :", conf["faithfulness_confidence"])
    print("Final Confidence:", fc, "-->", label)
    print()


Query           : Where should I park my rainy-day / emergency fund?
Answer          : Based on the provided documents, for a rainy-day/emergency fund, you should consider parking your money in a:

1. Savings account
2. Money market account
3. CDs (with a definite time period, such as 6 months, 1 year, or 2 years)

These options provide liquidity, allowing you to access your money qui
Retrieval Conf  : 0.3561
Faithfulness    : 0.9974
Final Confidence: 0.7409 --> HIGH - Answer confidently

Query           : Tax considerations for selling a property below appraised value to family?
Answer          : Based on the provided documents, selling a property below appraised value to a family member may result in trouble deducting losses on taxes, especially since it's a related person transaction. The state of Maryland has a transfer/recordation tax of 1.5% for both the buyer and seller, but it's uncle
Retrieval Conf  : 0.5365
Faithfulness    : 0.9979
Final Confidence: 0.8134 --> HIGH - Answer c

In [69]:
# ── Full Evaluation on all 648 queries (~30 min) ─────────────────────────────
# RUN THIS CELL to get NDCG@10, MRR, Recall@10 for the LLM Re-ranker pipeline
# Then the comparison table below shows all 4 models side by side

print("Running LLM re-ranker on all 648 queries (this takes ~30 min)...")
all_reranked = {}
for _, row in tqdm(test_queries_df.iterrows(), total=len(test_queries_df)):
    qid      = str(row["_id"])
    reranked = groq_rerank(str(row["text"]), list(hybrid_top20[qid].keys()),
                           corpus_id_to_text, client, top_k=10)
    all_reranked[qid] = reranked
    time.sleep(2.5)

evaluator    = pytrec_eval.RelevanceEvaluator(qrels_dict, {"ndcg_cut_10", "recip_rank", "recall_10"})
eval_results = evaluator.evaluate(all_reranked)
metrics = {
    "NDCG@10":   round(np.mean([v["ndcg_cut_10"] for v in eval_results.values()]), 4),
    "MRR":       round(np.mean([v["recip_rank"]  for v in eval_results.values()]), 4),
    "Recall@10": round(np.mean([v["recall_10"]   for v in eval_results.values()]), 4),
}
print("Evaluation done!")

# ── Comparison Table: All Models ─────────────────────────────────────────────
comparisons = [
    ("BM25 Baseline",            0.2169, 0.2706, 0.2784),
    ("MiniLM-Dense",             0.3687, 0.4451, 0.4413),
    ("Hybrid-Alpha (α=0.7)",     0.3791, 0.4606, 0.4473),
    ("LLM Re-ranker (Groq)",     metrics["NDCG@10"], metrics["MRR"], metrics["Recall@10"]),
]

print()
print("=" * 65)
print("           FULL PIPELINE COMPARISON — FiQA TEST SET")
print("=" * 65)
print(f"  {'Model':<28} {'NDCG@10':>8} {'MRR':>8} {'Recall@10':>10}  {'vs BM25':>8}")
print("-" * 65)
bm25_ndcg = 0.2169
for name, ndcg, mrr, recall in comparisons:
    pct = f"+{((ndcg - bm25_ndcg) / bm25_ndcg * 100):.1f}%" if ndcg != bm25_ndcg else "baseline"
    tag = "  ◀ BEST" if ndcg == max(c[1] for c in comparisons) else ""
    print(f"  {name:<28} {ndcg:>8.4f} {mrr:>8.4f} {recall:>10.4f}  {pct:>8}{tag}")
print("=" * 65)
print()
print("Metrics explanation:")
print("  NDCG@10   : Ranking quality  (industry target ≥ 0.45)")
print("  MRR       : Right doc at #1  (industry target ≥ 0.50)")
print("  Recall@10 : Coverage         (industry target ≥ 0.60)")

Running LLM re-ranker on all 648 queries (this takes ~30 min)...


100%|██████████| 648/648 [1:01:00<00:00,  5.65s/it]

Evaluation done!

           FULL PIPELINE COMPARISON — FiQA TEST SET
  Model                         NDCG@10      MRR  Recall@10   vs BM25
-----------------------------------------------------------------
  BM25 Baseline                  0.2169   0.2706     0.2784  baseline
  MiniLM-Dense                   0.3687   0.4451     0.4413    +70.0%
  Hybrid-Alpha (α=0.7)           0.3791   0.4606     0.4473    +74.8%  ◀ BEST
  LLM Re-ranker (Groq)           0.3602   0.4381     0.4266    +66.1%

Metrics explanation:
  NDCG@10   : Ranking quality  (industry target ≥ 0.45)
  MRR       : Right doc at #1  (industry target ≥ 0.50)
  Recall@10 : Coverage         (industry target ≥ 0.60)


In [ ]:
os.makedirs(FINRERANK_DIR, exist_ok=True)

demo_path = os.path.join(FINRERANK_DIR, "demo_results.json")
with open(demo_path, "w") as f:
    json.dump(demo_results, f, indent=2)
print("Saved:", demo_path)

try:
    metrics_out = {
        "model":       "LLM Re-ranker (Hybrid-Alpha -> Groq Llama-3.3-70B)",
        "dataset":     "FiQA Test",
        "num_queries": len(eval_results),
        **metrics
    }
    metrics_path = os.path.join(FINRERANK_DIR, "finrerank_metrics.json")
    with open(metrics_path, "w") as f:
        json.dump(metrics_out, f, indent=2)
    print("Saved:", metrics_path)
except NameError:
    print("Full metrics not saved (cell-12 was skipped)")

print("Done!")


## Pipeline B — Finance LLM Re-ranker (Palmyra-Fin-70B) + Groq Answer + NLI Confidence
**New approach:** Use `writer/palmyra-fin-70b-32k` (finance-domain LLM) for re-ranking instead of Groq Llama.  
Then Groq Llama 3.3 70B generates the answer. NLI DeBERTa scores faithfulness.  
**All existing cells kept intact above for comparison.**

In [70]:
# ── Finance-Aware Re-ranker using Mistral Large (mistralai/mistral-large-2411) ─
# Mistral Large is used in European banking/finance (BNP Paribas, Societe Generale)
# Strong at structured financial document analysis and relevance ranking

FIN_MODEL = "mistralai/mistral-large-2411"

fin_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

def finance_llm_rerank(query, candidates, corpus_id_to_text, client, model, top_k=5):
    """Re-rank using Mistral Large — finance-aware re-ranking."""
    lines = []
    for i, d in enumerate(candidates, 1):
        text = corpus_id_to_text.get(d, "")[:400].replace("\n", " ")
        lines.append(str(i) + ". " + text)
    n        = len(candidates)
    passages = "\n".join(lines)
    prompt = (
        "You are a financial document relevance expert specializing in investment, "
        "tax, and personal finance.\n\n"
        "Query: \"" + query + "\"\n\n"
        "Below are " + str(n) + " candidate financial passages. "
        "Return a JSON array of passage numbers (1-based) sorted MOST to LEAST relevant "
        "to the financial query. Include ALL " + str(n) + " numbers.\n\n"
        "Passages:\n" + passages + "\n\n"
        "Respond ONLY with a JSON array, e.g.: [3,1,5,2,4]"
    )
    try:
        r = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=200, temperature=0,
        )
        raw = r.choices[0].message.content.strip()
        m   = re.search(r"\[[\d,\s]+\]", raw)
        if m:
            ranking = json.loads(m.group())
            ranking = [x for x in ranking if 1 <= x <= n]
            seen    = set(ranking)
            for i in range(1, n + 1):
                if i not in seen:
                    ranking.append(i)
            return {candidates[idx - 1]: float(n - pos + 1)
                    for pos, idx in enumerate(ranking[:top_k], 1)}
    except Exception as e:
        print(f"  Mistral rerank error: {e}")
    return {d: float(n - i) for i, d in enumerate(candidates[:top_k])}


# ── Demo: Pipeline B on same 5 queries ───────────────────────────────────────
print("Pipeline B — Mistral Large (Finance Re-ranker) + Groq Answer + NLI Confidence")
print("=" * 70)

pipeline_b_results = []

for qid in SAMPLE_QIDS:
    query      = query_id_to_text.get(qid, "")
    candidates = list(hybrid_top20[qid].keys())

    # Step 1: Mistral Large re-ranks 20 -> top-5
    reranked_fin = finance_llm_rerank(query, candidates, corpus_id_to_text,
                                      fin_client, FIN_MODEL, top_k=5)
    top5_fin     = list(reranked_fin.keys())
    time.sleep(RATE_DELAY)

    # Step 2: Groq Llama generates answer (same as Pipeline A)
    answer_fin = groq_answer(query, top5_fin, corpus_id_to_text, client)
    time.sleep(RATE_DELAY)

    # Step 3: NLI DeBERTa confidence (same as Pipeline A)
    conf_fin  = compute_confidence(answer_fin, top5_fin, corpus_id_to_text,
                                   hybrid_top20[qid], nli_model)
    fc_fin    = conf_fin["final_confidence"]
    label_fin = ("HIGH - Answer confidently"   if fc_fin >= 0.7 else
                 "MEDIUM - Answer with caveat" if fc_fin >= 0.4 else
                 "LOW - Cannot answer reliably")

    pipeline_b_results.append({
        "query_id": qid, "query": query, "answer": answer_fin,
        "top5_docs": top5_fin, **conf_fin, "conf_label": label_fin
    })

    print(f"Query           : {query}")
    print(f"Answer          : {answer_fin[:300]}")
    print(f"Retrieval Conf  : {conf_fin['retrieval_confidence']}")
    print(f"Faithfulness    : {conf_fin['faithfulness_confidence']}")
    print(f"Final Confidence: {fc_fin} --> {label_fin}")
    print()

print("Pipeline B demo complete.")

Pipeline B — Mistral Large (Finance Re-ranker) + Groq Answer + NLI Confidence
Query           : Where should I park my rainy-day / emergency fund?
Answer          : Based on the provided documents, it is recommended to park your rainy-day/emergency fund in a:

1. Savings account
2. Money market account
3. CDs (with a short term, such as 6 months, 1 year, or 2 years)
4. Highly liquid exchange-traded products (with low volatility/drawdowns and insured)

These opt
Retrieval Conf  : 0.3567
Faithfulness    : 0.9969
Final Confidence: 0.7408 --> HIGH - Answer confidently

Query           : Tax considerations for selling a property below appraised value to family?
Answer          : Based on the provided documents, selling a property below appraised value to a family member may result in trouble deducting losses on taxes, especially since it's a related person transaction. The documents do not provide a clear answer on how the sale price versus appraised value affects tax consi
Retrieval Conf  

In [71]:
# ── Side-by-Side Comparison: Pipeline A vs Pipeline B ────────────────────────
# Pipeline A: Groq Llama 3.3 70B rerank → Groq answer → NLI
# Pipeline B: Palmyra-Fin-70B rerank    → Groq answer → NLI

print("=" * 90)
print("   PIPELINE COMPARISON — GROQ RERANK (A) vs FINANCE LLM RERANK (B) — 5 Demo Queries")
print("=" * 90)
print(f"  {'Query':<45} {'Pipeline A':>20} {'Pipeline B':>20}")
print(f"  {'':45} {'Groq Rerank':>20} {'Palmyra-Fin Rerank':>20}")
print(f"  {'':45} {'Conf / Label':>20} {'Conf / Label':>20}")
print("-" * 90)

a_confs = []
b_confs = []

for a, b in zip(demo_results, pipeline_b_results):
    q     = a["query"][:43] + ".." if len(a["query"]) > 43 else a["query"]
    a_fc  = a["final_confidence"]
    b_fc  = b["final_confidence"]
    a_lbl = "HIGH" if a_fc >= 0.7 else ("MED" if a_fc >= 0.4 else "LOW")
    b_lbl = "HIGH" if b_fc >= 0.7 else ("MED" if b_fc >= 0.4 else "LOW")
    winner = " ◀ B wins" if b_fc > a_fc else (" ◀ A wins" if a_fc > b_fc else " tie")
    a_confs.append(a_fc)
    b_confs.append(b_fc)
    print(f"  {q:<45} {a_fc:>6.4f} ({a_lbl:>4})   {b_fc:>6.4f} ({b_lbl:>4}){winner}")

print("-" * 90)
avg_a = round(sum(a_confs) / len(a_confs), 4)
avg_b = round(sum(b_confs) / len(b_confs), 4)
best  = "Pipeline B (Palmyra-Fin)" if avg_b > avg_a else "Pipeline A (Groq)"
print(f"  {'AVERAGE':<45} {avg_a:>6.4f}          {avg_b:>6.4f}        ◀ {best}")
print("=" * 90)

print()
print("Faithfulness breakdown (NLI DeBERTa):")
print(f"  {'Query':<45} {'A Faith':>10} {'B Faith':>10}")
print("-" * 68)
for a, b in zip(demo_results, pipeline_b_results):
    q = a["query"][:43] + ".." if len(a["query"]) > 43 else a["query"]
    print(f"  {q:<45} {a['faithfulness_confidence']:>10.4f} {b['faithfulness_confidence']:>10.4f}")

print()
print("Summary:")
print(f"  Pipeline A  — Groq Llama 3.3 70B rerank  → avg confidence: {avg_a}")
print(f"  Pipeline B  — Palmyra-Fin-70B rerank      → avg confidence: {avg_b}")
print(f"  Winner      : {best}")

   PIPELINE COMPARISON — GROQ RERANK (A) vs FINANCE LLM RERANK (B) — 5 Demo Queries
  Query                                                   Pipeline A           Pipeline B
                                                         Groq Rerank   Palmyra-Fin Rerank
                                                        Conf / Label         Conf / Label
------------------------------------------------------------------------------------------
  Where should I park my rainy-day / emergenc.. 0.7409 (HIGH)   0.7408 (HIGH) ◀ A wins
  Tax considerations for selling a property b.. 0.8134 (HIGH)   0.5221 ( MED) ◀ A wins
  Can the Delta be used to calculate the opti.. 0.1492 ( LOW)   0.3886 ( LOW) ◀ B wins
  Basic Algorithmic Trading Strategy            0.8392 (HIGH)   0.7852 (HIGH) ◀ A wins
  What does a high operating margin but a sma.. 0.7276 (HIGH)   0.7718 (HIGH) ◀ B wins
------------------------------------------------------------------------------------------
  AVERAGE                    

In [73]:
# ── Full Evaluation: Pipeline B (Mistral Large Re-ranker) on all 648 queries ─
# ~30-40 min — runs Mistral Large re-ranking on every test query
# Then shows NDCG@10, MRR, Recall@10 vs all previous models

print("Running Pipeline B (Mistral Large re-ranker) on all 648 queries (~30-40 min)...")
all_reranked_b = {}
for _, row in tqdm(test_queries_df.iterrows(), total=len(test_queries_df)):
    qid      = str(row["_id"])
    reranked = finance_llm_rerank(str(row["text"]), list(hybrid_top20[qid].keys()),
                              corpus_id_to_text, fin_client, FIN_MODEL, top_k=10)

    all_reranked_b[qid] = reranked
    time.sleep(2.5)

evaluator_b    = pytrec_eval.RelevanceEvaluator(qrels_dict, {"ndcg_cut_10", "recip_rank", "recall_10"})
eval_results_b = evaluator_b.evaluate(all_reranked_b)
metrics_b = {
    "NDCG@10":   round(np.mean([v["ndcg_cut_10"] for v in eval_results_b.values()]), 4),
    "MRR":       round(np.mean([v["recip_rank"]  for v in eval_results_b.values()]), 4),
    "Recall@10": round(np.mean([v["recall_10"]   for v in eval_results_b.values()]), 4),
}
print("Evaluation done!")

# ── Full Comparison Table: All Models ────────────────────────────────────────
all_comparisons = [
    ("BM25 Baseline",               0.2169, 0.2706, 0.2784),
    ("MiniLM-Dense",                0.3687, 0.4451, 0.4413),
    ("Hybrid-Alpha (α=0.7)",        0.3791, 0.4606, 0.4473),
    ("LLM Rerank A (Groq Llama)",   metrics["NDCG@10"],   metrics["MRR"],   metrics["Recall@10"]),
    ("LLM Rerank B (Mistral Large)",metrics_b["NDCG@10"], metrics_b["MRR"], metrics_b["Recall@10"]),
]

best_ndcg = max(c[1] for c in all_comparisons)
bm25_ndcg = 0.2169

print()
print("=" * 70)
print("        FULL PIPELINE COMPARISON — FiQA TEST SET (648 queries)")
print("=" * 70)
print(f"  {'Model':<32} {'NDCG@10':>8} {'MRR':>8} {'Recall@10':>10}  {'vs BM25':>8}")
print("-" * 70)
for name, ndcg, mrr, recall in all_comparisons:
    pct = f"+{((ndcg - bm25_ndcg) / bm25_ndcg * 100):.1f}%" if ndcg != bm25_ndcg else "baseline"
    tag = "  ◀ BEST" if ndcg == best_ndcg else ""
    print(f"  {name:<32} {ndcg:>8.4f} {mrr:>8.4f} {recall:>10.4f}  {pct:>8}{tag}")
print("=" * 70)
print()
print("Industry targets:  NDCG@10 ≥ 0.45 | MRR ≥ 0.50 | Recall@10 ≥ 0.60")

# Save Pipeline B metrics
os.makedirs(FINRERANK_DIR, exist_ok=True)
metrics_b_out = {
    "model":       "LLM Re-ranker B (Hybrid-Alpha -> Mistral Large -> Groq Answer)",
    "dataset":     "FiQA Test",
    "num_queries": len(eval_results_b),
    **metrics_b
}
with open(os.path.join(FINRERANK_DIR, "finrerank_b_metrics.json"), "w") as f:
    json.dump(metrics_b_out, f, indent=2)
print(f"\nSaved: finrerank/Results/finrerank_b_metrics.json")

Running Pipeline B (Mistral Large re-ranker) on all 648 queries (~30-40 min)...


100%|██████████| 648/648 [2:01:01<00:00, 11.21s/it]      

Evaluation done!

        FULL PIPELINE COMPARISON — FiQA TEST SET (648 queries)
  Model                             NDCG@10      MRR  Recall@10   vs BM25
----------------------------------------------------------------------
  BM25 Baseline                      0.2169   0.2706     0.2784  baseline
  MiniLM-Dense                       0.3687   0.4451     0.4413    +70.0%
  Hybrid-Alpha (α=0.7)               0.3791   0.4606     0.4473    +74.8%
  LLM Rerank A (Groq Llama)          0.3602   0.4381     0.4266    +66.1%
  LLM Rerank B (Mistral Large)       0.3885   0.4775     0.4485    +79.1%  ◀ BEST

Industry targets:  NDCG@10 ≥ 0.45 | MRR ≥ 0.50 | Recall@10 ≥ 0.60

Saved: finrerank/Results/finrerank_b_metrics.json
